In [122]:
import numpy as np
from dlfs.activation import Softmax
from dlfs.layers import DenseLayer
from dlfs.base import Module

## Attention mechanism in Transformers - simple example

In [123]:
X = np.array([[1, 2], # token 0
              [3, 4], # token 1
              [5, 6], # token 2
              [7, 8]]) # token 3

T, C = X.shape
print(f'The sequence X has {T} timesteps or {T} tokens.\nEach token is {C}-dimensonal')

The sequence X has 4 timesteps or 4 tokens.
Each token is 2-dimensonal


## Step 1. Averaging 

In [124]:
avg = np.zeros_like(X)

for t in range(X.shape[0]):
    x_prev = X[:t+1]    
    print(f'Current context for timestep {t}:\n{x_prev}')
    current_avg = np.mean(x_prev, axis=0)
    print(f'Current average for timestep {t}: {current_avg}\n----------------------------------------')
    avg[t] = current_avg

print(f'Result:\n{avg}')

Current context for timestep 0:
[[1 2]]
Current average for timestep 0: [1. 2.]
----------------------------------------
Current context for timestep 1:
[[1 2]
 [3 4]]
Current average for timestep 1: [2. 3.]
----------------------------------------
Current context for timestep 2:
[[1 2]
 [3 4]
 [5 6]]
Current average for timestep 2: [3. 4.]
----------------------------------------
Current context for timestep 3:
[[1 2]
 [3 4]
 [5 6]
 [7 8]]
Current average for timestep 3: [4. 5.]
----------------------------------------
Result:
[[1 2]
 [2 3]
 [3 4]
 [4 5]]


## Step 2. Averaging using lower triangular matrix

In [125]:
W = np.tril(np.ones((T, T))) # create lower triangular matrix of ones
print(f'Triangular matrix:\n{W}')

W = W / np.sum(W, axis=1, keepdims=True) # normalize rows of W so they add up to 1
print(f'Normalized across rows:\n{W}')

avg = np.matmul(W, X) # matrix product of W and X produces the same average
print(f'Result:\n{avg}')

Triangular matrix:
[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]
Normalized across rows:
[[1.         0.         0.         0.        ]
 [0.5        0.5        0.         0.        ]
 [0.33333333 0.33333333 0.33333333 0.        ]
 [0.25       0.25       0.25       0.25      ]]
Result:
[[1. 2.]
 [2. 3.]
 [3. 4.]
 [4. 5.]]


## Step 3. Adding Softmax to do the row normalization of triangular matrix

In [126]:
W = np.tril(np.ones((T, T))) # create lower triangular matrix of ones
print(f'Triangular matrix:\n{W}\n--------------------')

mask = W == 0 # create mask, find elements which are equal to 0
print(f'Mask:\n{mask}\n--------------------')

W[mask] = -np.inf # set elements which are 0 to -inf
print(f'Masked W:\n{W}\n--------------------')

soft = Softmax()
soft.forward(W) # apply softmax to rows, effectively normalizing rows to sum to 1
W = soft.output
print(f'Softmax W:\n{W}\n--------------------')

avg = np.matmul(W, X) # matrix product of W and X produces the same average
print(f'Result:\n{avg}')

Triangular matrix:
[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]
--------------------
Mask:
[[False  True  True  True]
 [False False  True  True]
 [False False False  True]
 [False False False False]]
--------------------
Masked W:
[[  1. -inf -inf -inf]
 [  1.   1. -inf -inf]
 [  1.   1.   1. -inf]
 [  1.   1.   1.   1.]]
--------------------
Softmax W:
[[1.         0.         0.         0.        ]
 [0.5        0.5        0.         0.        ]
 [0.33333333 0.33333333 0.33333333 0.        ]
 [0.25       0.25       0.25       0.25      ]]
--------------------
Result:
[[1. 2.]
 [2. 3.]
 [3. 4.]
 [4. 5.]]


***
## Whole attention process - single sequence

- given an input sequence $X \in \mathbb{R}^{T \times d}$ where $T$ is the number of timesteps (sequence length) and $d$ is the embedding dimension of a single sequence element, matrices $Q$ (query), $K$ (key) and $V$ (value) are computed by applying linear transformations:

$$Q = XW^{Q} \quad \quad K = XW^{K} \quad \quad V = XW^{V} \quad \quad W^{Q}, W^{K}, W^{V} \in \mathbb{R}^{d \times h}$$

- $Q, K, V \in \mathbb{R}^{T \times h}$ are then used to calculate attention given by the following expression:

$$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{h}} \right)V$$

- $QK^T$ serves as similarity score between every pair of query and key vectors and is divided elementwise by $\sqrt{h}$ for stability

- if attention is used in autoregressive manner (decoder only, text generation) masking must be applied to $QK^T$ to prevent tokens from attending to future tokens (masking code example provided above)

- softmax is applied to similarity scores to calculate attention scores (effectively weights for each token pair, normalized to 1)

- attention scores are multiplied with value matrix $V$ producing final output, obtaining contextualized representation for each token

***

## Takeaways - attention simplified

- $Q$ represents the token that is looking for information

- $K$ represents the token providing information

- $V$ represents the actual content that will be passed when the token attends to others

- $W$ is the matrix of attention weights that determines how much focus each token should give to each other token, and it is computed using the similarity between $Q$ and $K$, $W=\text{softmax} \left( \frac{QK^T}{\sqrt{h}} \right)$.

- this concept is also refered to as self-attention because a sequence is attending to itself

***

## Whole attention process - multiple sequences, batches

- batch refers to a collection of samples, for example a batch of input sequences is a tensor $\mathbf{X} \in \mathbb{R}^{B \times T \times d}$ meaning in $\mathbf{X}$ there are $B$ samples, each sample has sequence length $T$ and each token is $d$ dimensional

- doing matrix multiplication with respect to batches $Q, K, V$ will be in $\mathbb{R}^{B \times T \times h}$

- doing $W=\text{softmax} \left( \frac{QK^T}{\sqrt{h}} \right)$ results in $W \in \mathbb{R}^{B \times T \times T}$ (transposing $K$ means swapping last two axes from $T \times h$ to $h \times T$)

- final output $WV$ is in $\mathbb{R}^{B \times T \times h}$

***

# Single attention

- attention is often split into multiple heads where one head is a single module doing the attention operations

In [127]:
class SingleAttentionHead(Module):

    def __init__(self, input_size: int, head_size: int, dropout=0.1, use_mask=False) -> None:
        """
        A single head of attention for computing the attention mechanism in models like Transformers.

        Parameters
        ----------
        input_size : int
            The size of the input features. This is the dimension of the input to the attention mechanism (token embedding size).
            
        head_size : int
            The size of each attention head, dimensionality of the query, key, and value vectors.
            
        dropout : float, optional, default=0.1
            The dropout rate applied to the attention weights during training.
            
        use_mask : bool, optional, default=False
            If True, applies a mask to ensure that the attention mechanism cannot attend to future tokens.
        """
        self.key = DenseLayer(input_size, head_size)
        self.query = DenseLayer(input_size, head_size)
        self.value = DenseLayer(input_size, head_size)
        self.softmax = Softmax()
        self.dropout = DropoutLayer(dropout)

        self.normalize_factor = head_size**0.5
        self.use_mask = use_mask

    def forward(self, query_input: np.ndarray, context_input: np.ndarray, training: bool = False) -> None:
        """
        Forward pass for the Single Attention Head. Creates output attribute.

        Parameters
        ----------
        query_input : np.ndarray
            Input array of shape `(batch_size, seq_len, input_size)` used in computing query matrix.

        context_input : np.ndarray
            Input array of shape `(batch_size, seq_len, input_size)` used in computing key and value matrices.
            In self-attention, this array will be the same as `query_input`, but in encoder-decoder cross-attention, 
            the `context_input` comes from the encoder.

        training : bool, default=False
            A flag indicating whether the model is in training mode. If True, dropout is applied to the attention
            weights.

        Returns
        -------
        None
        """

        # Compute Q, K and V matrices
        self.query.forward(query_input)
        self.key.forward(context_input)
        self.value.forward(context_input)

        self.q = self.query.output
        self.k = self.key.output
        self.v = self.value.output
        
        # Compute similarity scores (unnormalized attention scores) between all vector pairs of Q and K 
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor

        if self.use_mask:
            B, T, _ = self.q.shape
            # Create lower triangular mask matrix
            mask = np.tril(np.ones((T, T), dtype=bool))
            # Apply mask to future tokens
            self.w = np.where(mask[None, :, :], self.w, -np.inf)

        # Compute attention scores
        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.attn_weights = self.softmax.output.copy()

        # Add dropout
        self.dropout.forward(self.w, training)
        self.w = self.dropout.output

        # Compute final output
        self.output = np.matmul(self.w, self.v)

    def backward(self, delta: np.ndarray) -> None:
        """
        Backward pass for the Single Attention Head. Creates dinputs gradient attributes.

        Parameters
        ----------
        delta : np.ndarray
            Gradient array of shape `(batch_size, seq_len, head_size)` used in computing query matrix.

        Returns
        -------
        None
        """
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        self.dropout.backward(d_w)
        d_w = self.dropout.dinputs

        self.softmax.backward(d_w)
        d_w = self.softmax.dinputs

        if self.use_mask:
            B, T, _ = d_w.shape
            mask = np.tril(np.ones((T, T), dtype=bool))
            d_w = d_w * mask[None, :, :]

        d_q = np.matmul(d_w, self.k) / self.normalize_factor
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q) / self.normalize_factor

        d_v_input = np.matmul(self.attn_weights.transpose(0, 2, 1), delta)

        self.key.backward(d_k)
        self.query.backward(d_q)
        self.value.backward(d_v_input)

        self.dinputs_query = self.query.dinputs
        self.dinputs_context = self.key.dinputs + self.value.dinputs

# Multihead Attention

- the results from each attention head are concatenated into one output

In [128]:
class MultiHeadAttention(Module):

    def __init__(self, n_embed: int, n_heads: int, dropout: float = 0.1, use_mask: bool = False) -> None:
        """
        Multi Head Attention module which consists of multiple `SingleAttentionHead` objects.

        Parameters
        ----------
        n_embed : int
            Dimensionality of a single embedding token.

        n_heads : int
            Number of single attention heads to create in the module.
            `n_embed` must be divisible by this number.

        dropout : float, default=0.1
            The dropout rate applied to the attention weights during training.
            
        use_mask : bool, optional, default=False
            If True, applies a mask to ensure that the attention mechanism cannot attend to future tokens.
        """
        # Assertion to ensure n_embed is divisible by n_heads
        assert n_embed % n_heads == 0, f"n_embed ({n_embed}) must be divisible by n_heads ({n_heads})."
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads

        self.attention_heads = [
            SingleAttentionHead(n_embed, self.head_size, dropout, use_mask)
            for _ in range(n_heads)
        ]

        self.output_dense = DenseLayer(n_embed, n_embed)

        self.dropout = DropoutLayer(dropout)

    def forward(self, query_input: np.ndarray, context_input: np.ndarray = None, training: bool = False) -> None:
        """
        Forward pass for the MultiHeadAttention. Creates output attribute.

        Parameters
        ----------
        query_input : np.ndarray
            Input array of shape `(batch_size, seq_len, input_size)` used in computing query matrix.

        context_input : np.ndarray
            Input array of shape `(batch_size, seq_len, input_size)` used in computing key and value matrices.
            In self-attention, this array will be the same as `query_input`, but in encoder-decoder cross-attention, 
            the `context_input` comes from the encoder.

        training : bool, default=False
            A flag indicating whether the model is in training mode. If True, dropout is applied to the attention
            weights.

        Returns
        -------
        None
        """
        # Check if context_input is given
        self.is_cross_attention = context_input is not None
        if context_input is None:
            # If there is no context input, query input gets passed to heads
            context_input = query_input

        head_outputs = []
        for i, head in enumerate(self.attention_heads):
            head.forward(query_input, context_input, training)
            head_outputs.append(head.output)

        # Concatenate head results
        concatenated_output = np.concatenate(head_outputs, axis=-1)
        self.output_dense.forward(concatenated_output)
        self.dropout.forward(self.output_dense.output, training)
        self.output = self.dropout.output

    def backward(self, delta: np.ndarray) -> None:
        """
        Backward pass for the Multi Head Attention. Creates dinputs gradient attributes.

        Parameters
        ----------
        delta : np.ndarray
            Gradient array of shape `(batch_size, seq_len, input_size)`.

        Returns
        -------
        None
        """

        self.dropout.backward(delta)

        self.output_dense.backward(self.dropout.dinputs)

        d_concatenated_output = self.output_dense.dinputs

        d_head_outputs = np.split(d_concatenated_output, self.n_heads, axis=-1)

        self.attention_heads[0].backward(d_head_outputs[0])
        self.dinputs_query = np.zeros_like(self.attention_heads[0].dinputs_query)
        self.dinputs_context = np.zeros_like(self.attention_heads[0].dinputs_context)

        self.dinputs_query += self.attention_heads[0].dinputs_query
        self.dinputs_context += self.attention_heads[0].dinputs_context

        for i, head in enumerate(self.attention_heads[1:], start=1):
            head.backward(d_head_outputs[i])
            self.dinputs_query += self.attention_heads[i].dinputs_query
            self.dinputs_context += self.attention_heads[i].dinputs_context

        #self.dinputs_query /= self.n_heads
        #self.dinputs_context /= self.n_heads

        if self.is_cross_attention:
            self.dinputs = None
        else:
            self.dinputs = self.dinputs_query + self.dinputs_context